# 04 - Multi-Dimensional Evaluation & Experiments
**Quantifying Accuracy vs. Dimensionality vs. Storage vs. Latency**

We evaluate the fine-tuned Matryoshka model across:
- **768d:** Full representation.
- **512d:** Minor compression.
- **256d:** Target Pareto sweet spot.
- **128d:** Maximum safe compression (83.3% storage savings).
- **64d:** Stress-test dimension to observe the capacity cliff.


In [34]:
import os

# 1. Clone only if not already cloned
if not os.path.exists("matryoshka-domain-rag") and not os.path.exists("src"):
    !git clone https://github.com/premsaipusapati-debug/matryoshka-domain-rag.git
    %cd matryoshka-domain-rag
elif os.path.exists("matryoshka-domain-rag"):
    %cd matryoshka-domain-rag

# 2. Pull any recent changes from GitHub
!git pull

# 3. Install requirements
!pip install -q -r requirements.txt


Already up to date.


In [35]:
# 1. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# 2. Check where the model is located
potential_paths = [
    "/content/drive/MyDrive/Matryoshka-RAG/bge-scifact-matryoshka",
    "/content/matryoshka-domain-rag/output/bge-scifact-matryoshka",
    "/content/output/bge-scifact-matryoshka",
]

checkpoint_dir = None
for p in potential_paths:
    if os.path.exists(p):
        checkpoint_dir = p
        break

print(f"✅ Found fine-tuned model at: {checkpoint_dir}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found fine-tuned model at: None


In [36]:
# Search your Google Drive and Colab for the saved model
!find /content/drive/MyDrive -name "*bge-scifact*" 2>/dev/null
!find /content -name "*bge-scifact*" 2>/dev/null


In [37]:
import sys
import os

if os.path.basename(os.getcwd()) == "notebooks":
    sys.path.append(os.path.abspath(".."))
else:
    sys.path.append(os.getcwd())

import yaml
import pandas as pd
from src.data_loader import load_scifact_raw, get_eval_data
from src.model import load_embedding_model
from src.evaluate import run_dimensional_benchmarks

config_path = "../configs/config.yaml" if os.path.exists("../configs/config.yaml")  else "configs/config.yaml"
with open(config_path, "r") as f:
  config = yaml.safe_load(f)

# Locate trained model
checkpoint_dir = '/content/drive/MyDrive/Matryoshka-RAG/bge-scifact-matryoshka'
if not os.path.exists(checkpoint_dir):
  checkpoint_dir = "/content/bge-scifact-matryoshka"

# Fallback to the base model configuration if local checkpoint doesn't exist
if not os.path.exists(checkpoint_dir):
    checkpoint_dir = config["model"]["base_model"]
    print(f"⚠️ Fine-tuned model checkpoint not found. Falling back to base model: {checkpoint_dir}")
else:
    print(f"Loading fine-tuned MRL model from: {checkpoint_dir}")

mrl_model = load_embedding_model(checkpoint_dir, max_seq_length=config["model"]["max_seq_length"])


⚠️ Fine-tuned model checkpoint not found. Falling back to base model: BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [38]:
raw_data = load_scifact_raw(config["dataset"]["name"])
corpus_dict, queries_dict, qrels_dict = get_eval_data(raw_data, split="test")

print(f"Loaded {len(corpus_dict)} documents and {len(queries_dict)} test queries.")

Loaded 5183 documents and 300 test queries.


In [39]:
dimensions = config["matryoshka"]["eval_dimensions"] #[768, 512, 256, 128, 64]
print(f"Benchmarking dimensions: {dimensions}")

results_df = run_dimensional_benchmarks(
    corpus_dict=corpus_dict,
    queries_dict=queries_dict,
    qrels_dict=qrels_dict,
    model=mrl_model,
    dimensions=dimensions,
    top_k=config["evaluation"]["top_k"]
)

print("\n--- Final Experimental Results Matrix ---")
display(results_df)

#Save results
out_file = "../output/ mrl_dimensions_results.csv" if os.path.exists("../output") else "output/mrl_dimensional_results.csv"
results_df.to_csv(out_file, index=False)
print(f"\nSaved benchmark results to: {out_file}")

Benchmarking dimensions: [768, 512, 256, 128, 64]


Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


--- Final Experimental Results Matrix ---


,Dimension,Recall@10,MRR@10,nDCG@10,Storage_per_1M_MB,Storage_Savings_%,Avg_Latency_ms
0,768,0.8767,0.7004,0.7376,2929.7,0.0,0.090
1,512,0.8700,0.6978,0.7327,1953.1,33.3,0.110
2,256,0.8400,0.6702,0.7062,976.6,66.7,0.242
3,128,0.8100,0.6333,0.6681,488.3,83.3,0.159
4,64,0.7100,0.5329,0.5664,244.1,91.7,0.080


OSError: Cannot save file into a non-existent directory: 'output'